In [16]:
# Python ≥3.5 is required
import sys
assert sys.version_info >= (3, 5)

# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# TensorFlow ≥2.0 is required
import tensorflow as tf
from tensorflow import keras
assert tf.__version__ >= "2.0"

# Common imports
import numpy as np
import os

# to make this notebook's output stable across runs
np.random.seed(42)
tf.random.set_seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

In [13]:
#The goal of this project is to use a Generative Adversarial Network (GAN) to generate supplemental data for histopathological brain tumor samples, providing a solution to the relative scarcity of publicly available brain tumor data for machine learning and other
# scientific endeavors.

In [19]:
OPENSLIDE_PATH = r'C:\Users\Aihara\Desktop\Temporary Jupter Stuff\openslide-win64-20221217\openslide-win64-20221217\bin'

import os
if hasattr(os, 'add_dll_directory'):
    # Python >= 3.8 on Windows
    with os.add_dll_directory(OPENSLIDE_PATH):
        import openslide
else:
    import openslide

ModuleNotFoundError: No module named 'openslide'

In [24]:
import os
with os.add_dll_directory(OPENSLIDE_PATH):
    import openslide

ModuleNotFoundError: No module named 'openslide'

In [ ]:
from sklearn.model_selection import train_test_split 

In [ ]:
# TO-DO LIST
#
# Import .csv of the data, identify which 3/4 types of brain tumors are most common.
#      Related --> find out how to import .csv as pandas dataframe for easier working.
# Determine how/where to save data????? The images are huge, 3TiB in total. Even just the ones we care about will take up a huge amount of space.
# Determine and implement the best quantitative evaluation metric for this project.

In [ ]:
X = # Here we load in the data of the selected brain tumor types.
y = # Here we load in the labels of the selected brain tumor types.

X_train_full, y_train_full, X_test, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)
X_train, y_train, X_valid, y_valid =  train_test_split(X_train_full, y_train_full, test_size = 0.5, random_state = 42)

In [14]:
# DCGAN code taken from HOA #8 - will tweak as necessary to serve as a basis for this work.
# The DCGAN seemed to have better initial and overall performance as compared to the standard GAN, so we use it as the baseline here.

In [ ]:
tf.random.set_seed(42)
np.random.seed(42)

codings_size = 100

generator = keras.models.Sequential([
    keras.layers.Dense(7 * 7 * 128, input_shape=[codings_size]),
    keras.layers.Reshape([7, 7, 128]),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(64, kernel_size=5, strides=2, padding="SAME",
                                 activation="selu"),
    keras.layers.BatchNormalization(),
    keras.layers.Conv2DTranspose(1, kernel_size=5, strides=2, padding="SAME",
                                 activation="tanh"),
])
discriminator = keras.models.Sequential([
    keras.layers.Conv2D(64, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2),
                        input_shape=[28, 28, 1]),
    keras.layers.Dropout(0.4),
    keras.layers.Conv2D(128, kernel_size=5, strides=2, padding="SAME",
                        activation=keras.layers.LeakyReLU(0.2)),
    keras.layers.Dropout(0.4),
    keras.layers.Flatten(),
    keras.layers.Dense(1, activation="sigmoid")
])
gan = keras.models.Sequential([generator, discriminator])

In [ ]:
# Since the discriminator is a binary classifier (real/fake image), we can set the loss to binary_crossentropy.
# Similarly, since the GAN as a whole is a binary classifier, we can do the same.
# 
# Once the GAN is created, we must appropriately compile/determine which parts of the model are trainable or not.
# Since "trainable" is only taken into account while compiling, we first compile the discriminator assuming it will be trained. Then, we switch it so that it cannot be trained once the complete GAN is compiled.  
# This way, when we train the GAN, we can separate the training of the discriminator and the generator during each pass - the discriminator is first trained on 

In [ ]:
discriminator.compile(loss="binary_crossentropy", optimizer="rmsprop")
discriminator.trainable = False
gan.compile(loss="binary_crossentropy", optimizer="rmsprop")

In [ ]:
# In order to train a GAN, we must use a custom training loop similar to the one defined in the HOA/textbook.
# To do this, the training is split into two phases for each training pass/epoch.
# In phase 1, the discriminator is trained on a a combined dataset of real images and images built by the generator using Gaussian noise as input.
# In phase 2, the generator is trained by feeding Gaussian noise to the GAN itself. Now, the images from the generator are tested using the newly-trained discriminator from phase 1. 

In [15]:
batch_size = 32
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(1000)
dataset = dataset.batch(batch_size, drop_remainder=True).prefetch(1)

NameError: name 'X_train' is not defined

In [ ]:
def train_gan(gan, dataset, batch_size, codings_size, n_epochs=50):
    generator, discriminator = gan.layers
    for epoch in range(n_epochs):
        print("Epoch {}/{}".format(epoch + 1, n_epochs))
        for X_batch in dataset:
            
            # PHASE 1 - TRAINING THE DISCRIMINATOR
            
            # Generating fake images for training the discriminator
            noise = tf.random.normal(shape=[batch_size, codings_size])
            generated_images = generator(noise)
            
            # Combined dataset of both real and newly generated fake images 
            X_fake_and_real = tf.concat([generated_images, X_batch], axis=0)
            y1 = tf.constant([[0.]] * batch_size + [[1.]] * batch_size)
            
            # Actually training the discriminator on the above data.
            discriminator.trainable = True
            discriminator.train_on_batch(X_fake_and_real, y1)
            
            # PHASE 2 - TRAINING THE GENERATOR
            
            # Generating noise to feed into the GAN
            noise = tf.random.normal(shape=[batch_size, codings_size])
            y2 = tf.constant([[1.]] * batch_size)
            
            # Actually training the generator by having the newly-trained discriminator classify its attempts as real/fake images. 
            discriminator.trainable = False
            gan.train_on_batch(noise, y2)         
        
        # Maybe we can get rid of these?
        plot_multiple_images(generated_images, 8)
        plt.show()